In [4]:
#1.1
import pandas as pd
import re

# Load dataset
df = pd.read_json("C:\\Users\\kgour\\OneDrive\\Documents\\School\\ITNP\\Research-Document-Mining\\datasets\\research_abstracts.json")

print("Original dataset shape:", df.shape)

# Remove irrelevant DOI column
df = df.drop(columns=["DOI"])

# Remove rows with missing abstracts or labels
df = df.dropna(subset=["Abstract", "Label"])

# Remove duplicate records
df = df.drop_duplicates()

# Clean research abstracts
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["Abstract"] = df["Abstract"].apply(clean_text)

print("\nPreprocessed dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nFirst five cleaned abstracts:")
print(df["Abstract"].head())


#1.2

from sklearn.feature_extraction.text import TfidfVectorizer

# Identify input and target columns
X = df["Abstract"]
y = df["Label"]

# Convert abstracts into numerical TF-IDF features
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(X)

print("Input feature:", "Abstract")
print("Target feature:", "Label")
print("TF-IDF feature matrix shape:", X_tfidf.shape)
print("Number of unique labels:", y.nunique())
print("\nLabel distribution:")
print(y.value_counts())

#1.3

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create baseline classification model
model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train model
model.fit(X_train, y_train)

# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate model
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("Testing Accuracy:", accuracy_score(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))


Original dataset shape: (1138, 3)

Preprocessed dataset shape: (1125, 2)

Missing values:
Abstract    0
Label       0
dtype: int64

First five cleaned abstracts:
0    forward looking forecasting of the inflation r...
1    our study examined the disaggregation of infla...
2    this paper develops a dynamic factor model tha...
3    in this paper i use a simple sir macro model t...
4    backgroundgame theory is useful for identifyin...
Name: Abstract, dtype: object
Input feature: Abstract
Target feature: Label
TF-IDF feature matrix shape: (1125, 5000)
Number of unique labels: 2

Label distribution:
Label
classical    890
ai           235
Name: count, dtype: int64
Training Accuracy: 0.8822222222222222
Testing Accuracy: 0.8222222222222222

Classification Report:
              precision    recall  f1-score   support

          ai       1.00      0.15      0.26        47
   classical       0.82      1.00      0.90       178

    accuracy                           0.82       225
   macro avg  